# Debiased Two-Stage Hierarchical XGBoost on CICIoT2023
This notebook documents the balanced extraction, debiasing, and two-stage hierarchical model training for IoT network intrusion detection.

In [ ]:
import os
import glob
import time
import joblib
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Define 23 debiased flow dynamics features
DEBIASED_FEATURES = [
    'Header_Length', 'Time_To_Live', 'Rate',
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number',
    'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count',
    'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance'
]

print(f"Using {len(DEBIASED_FEATURES)} debiased flow features:", DEBIASED_FEATURES)

## 1. Load Trained Pipeline Artifacts and Run Sample Inference

In [ ]:
model_path = os.path.join("models", "ciciot2023_debiased_hierarchical.joblib")
artifact = joblib.load(model_path)

binary_model = artifact["binary_model"]
multiclass_model = artifact["multiclass_model"]
scaler = artifact["scaler"]
le = artifact["label_encoder"]

print("Classes:", le.classes_)
print("Binary Model Accuracy:", artifact["metrics"]["binary_accuracy"])
print("Multi-Class Model Accuracy:", artifact["metrics"]["multiclass_accuracy"])

## 2. Test Single-Flow Inference

In [ ]:
sample_record = {
    'Header_Length': 54.0,
    'Time_To_Live': 64.0,
    'Rate': 12.5,
    'fin_flag_number': 0,
    'syn_flag_number': 1,
    'rst_flag_number': 0,
    'psh_flag_number': 0,
    'ack_flag_number': 0,
    'ece_flag_number': 0,
    'cwr_flag_number': 0,
    'ack_count': 0,
    'syn_count': 1,
    'fin_count': 0,
    'rst_count': 0,
    'Tot sum': 1500.0,
    'Min': 40.0,
    'Max': 1500.0,
    'AVG': 500.0,
    'Std': 120.0,
    'Tot size': 1500.0,
    'IAT': 0.05,
    'Number': 5,
    'Variance': 14400.0
}

X_sample = pd.DataFrame([[sample_record.get(f, 0.0) for f in DEBIASED_FEATURES]], columns=DEBIASED_FEATURES)
X_scaled = scaler.transform(X_sample)

# Stage 1: Binary Filter
bin_pred = binary_model.predict(X_scaled)[0]
bin_prob = binary_model.predict_proba(X_scaled)[0]
print(f"Stage 1 Binary Prediction: {'Attack' if bin_pred == 1 else 'Benign'} (Prob: {bin_prob})")

# Stage 2: Multi-class Attribution
mc_pred_idx = multiclass_model.predict(X_scaled)[0]
mc_probs = multiclass_model.predict_proba(X_scaled)[0]
pred_label = le.inverse_transform([mc_pred_idx])[0]
print(f"Stage 2 Attack Category: {pred_label} (Confidence: {mc_probs[mc_pred_idx]:.4f})")